# SciGeoGuard-X baseline and ablation pilot

Run this in a fresh Google Colab CPU runtime. It clones a fixed repository commit, installs the importable package, runs the three conservative non-LLM baselines and the two ablations, scores predictions against the **generated** frozen v3 reference in a separate command, records environment metadata, and downloads a ZIP.

The baseline prediction script reads only the case ID and workflow input fields. It does not read reference labels. The scoring command reads the frozen reference only after predictions have been written. This experiment is a reproducibility/comparison pilot; it does not provide independent scientific accuracy or human validation.

In [ ]:
import os, subprocess, sys, json, platform, datetime, shutil
from pathlib import Path
from google.colab import files

REPO_URL = "https://github.com/bnssaanirudh/SciGeoGuard.git"
PINNED_COMMIT = "a37626dc6d6c4aa377ec0bce567246672787506e"
REPO = Path("/content/SciGeoGuard")
OUT = Path("/content/SciGeoGuard_Pilot_Results")

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == PINNED_COMMIT, (actual, PINNED_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "."], cwd=REPO, check=True)
OUT.mkdir(parents=True, exist_ok=True)
print({"commit": actual, "python": sys.version, "platform": platform.platform()})

In [ ]:
cases = REPO / "results/history/v6/Blinded_180_cases.csv"
reference = REPO / "results/history/v6/Frozen_v3_reference.csv"
baseline_dir = OUT / "baselines"
ablation_dir = OUT / "ablations"
baseline_dir.mkdir(parents=True, exist_ok=True)
ablation_dir.mkdir(parents=True, exist_ok=True)

# Prediction runs read only the allowed input fields.
subprocess.run([sys.executable, "experiments/baselines/run_baselines.py", "--cases", str(cases), "--out", str(baseline_dir / "predictions.csv")], cwd=REPO, check=True)
subprocess.run([sys.executable, "experiments/ablations/run_ablations.py", "--cases", str(cases), "--out", str(ablation_dir / "predictions.csv")], cwd=REPO, check=True)
print("Prediction files saved before scoring:", baseline_dir / "predictions.csv", ablation_dir / "predictions.csv")

In [ ]:
# Score only after predictions have been written. Targets are the generated frozen reference.
subprocess.run([sys.executable, "experiments/baselines/score_predictions.py", "--predictions", str(baseline_dir / "predictions.csv"), "--reference", str(reference), "--out", str(baseline_dir / "metrics.json")], cwd=REPO, check=True)
subprocess.run([sys.executable, "experiments/baselines/score_predictions.py", "--predictions", str(ablation_dir / "predictions.csv"), "--reference", str(reference), "--out", str(ablation_dir / "metrics.json")], cwd=REPO, check=True)

In [ ]:
# Record runtime details and input/code hashes for the downloaded result package.
import hashlib
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
manifest={
 "created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
 "repository":REPO_URL,"commit":PINNED_COMMIT,"python":sys.version,"platform":platform.platform(),
 "prediction_input_sha256":sha256(cases),
 "generated_reference_sha256_used_for_scoring":sha256(reference),
 "scope":"conservative baseline and ablation pilot; generated-reference scoring only",
 "limitations":["Not independent human ground truth.","Baselines never issue PROVEN and have intentionally limited rule coverage.","No confidence intervals: cases may share workflow templates.","Ablations measure consistency with author-generated labels, not scientific validity." ]
}
(OUT/"environment_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
(OUT/"README.txt").write_text("Results from pinned SciGeoGuard-X baseline/ablation pilot. Predictions are stored separately from metrics. Labels are generated reference only. See environment_manifest.json for commit, versions, and hashes.\n")
zip_path=Path("/content/SciGeoGuard_Pilot_Results.zip")
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")),"zip",root_dir=OUT)
print("ZIP ready:",zip_path,zip_path.stat().st_size,"bytes")
files.download(str(zip_path))